# Computing the spreads and the mid prices

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Helper to create inputs with increment/decrement buttons
def create_price_control(description, initial_value):
    val_text = widgets.FloatText(value=initial_value, description=description, step=0.01, layout=widgets.Layout(width="220px"))
    slider = widgets.FloatSlider(value=initial_value, min=0.0, max=100.0, step=0.01, readout=False, layout=widgets.Layout(width="400px"))
    
    btn_minus_01 = widgets.Button(description="-0.1", layout=widgets.Layout(width="50px", height="36px", button_style="warning"))
    btn_minus_001 = widgets.Button(description="-0.01", layout=widgets.Layout(width="60px", button_style="info"))
    btn_plus_001 = widgets.Button(description="+0.01", layout=widgets.Layout(width="60px", button_style="info"))
    btn_plus_01 = widgets.Button(description="+0.1", layout=widgets.Layout(width="50px", height="36px", button_style="warning"))
    
    # Link text box and slider
    widgets.jslink((val_text, "value"), (slider, "value"))
    
    def on_minus_01(b): val_text.value -= 0.1
    def on_minus_001(b): val_text.value -= 0.01
    def on_plus_001(b): val_text.value += 0.01
    def on_plus_01(b): val_text.value += 0.1
    
    btn_minus_01.on_click(on_minus_01)
    btn_minus_001.on_click(on_minus_001)
    btn_plus_001.on_click(on_plus_001)
    btn_plus_01.on_click(on_plus_01)
    
    controls = widgets.HBox([btn_minus_01, btn_minus_001, val_text, btn_plus_001, btn_plus_01], layout=widgets.Layout(align_items="center"))
    return val_text, slider, widgets.VBox([controls, slider])

ask_val, ask_slider, ask_ui = create_price_control("Ask Price:", 32.465)
bid_val, bid_slider, bid_ui = create_price_control("Bid Price:", 32.315)
offered_val, offered_slider, offered_ui = create_price_control("Offered Price:", 32.500)

# Toggles
buy_sell_toggle = widgets.ToggleButtons(options=["Buy", "Sell"], description="Action:", button_style="success")
precise_mode_toggle = widgets.ToggleButton(value=False, description="Precise Mode", icon="crosshairs", button_style="primary")

def on_precise_toggle(change):
    if change["new"]:
        # ON: very slow sliding around current value
        for s, v in [(ask_slider, ask_val), (bid_slider, bid_val), (offered_slider, offered_val)]:
            s.min = max(0.0, v.value - 0.2)
            s.max = v.value + 0.2
            s.step = 0.001
    else:
        # OFF: wide sliding range
        for s, v in [(ask_slider, ask_val), (bid_slider, bid_val), (offered_slider, offered_val)]:
            s.min = 0.0
            s.max = 100.0
            s.step = 0.01

precise_mode_toggle.observe(on_precise_toggle, names="value")

# Keep precise range centered if we manually edit the text box while in precise mode
def center_slider_if_precise(change, slider):
    if precise_mode_toggle.value:
        slider.min = max(0.0, change["new"] - 0.2)
        slider.max = change["new"] + 0.2

ask_val.observe(lambda c: center_slider_if_precise(c, ask_slider), names="value")
bid_val.observe(lambda c: center_slider_if_precise(c, bid_slider), names="value")
offered_val.observe(lambda c: center_slider_if_precise(c, offered_slider), names="value")

out = widgets.Output()

def update_dashboard(*args):
    out.clear_output(wait=True)
    with out:
        ask = ask_val.value
        bid = bid_val.value
        offered = offered_val.value
        action = buy_sell_toggle.value
        
        if action == "Buy":
            buy_sell_toggle.button_style = "success"
            offered_val.description = "Offered Ask:"
        else:
            buy_sell_toggle.button_style = "danger"
            offered_val.description = "Offered Bid:"
            
        if ask <= bid:
            display(HTML("<b style=\"color:red;\">Warning: Ask price must be greater than Bid price.</b>"))
            return
            
        mid = (ask + bid) / 2
        
        if action == "Buy":
            # Buying -> we look at Ask spread and Offered Ask spread
            mkt_spread_bps = (ask - mid) / mid * 10000
            offered_spread_bps = (offered - mid) / mid * 10000
            mkt_label = "Market Ask Spread"
            off_label = "Your Offered Ask Spread"
        else:
            # Selling -> we look at Bid spread and Offered Bid spread
            mkt_spread_bps = (mid - bid) / mid * 10000
            offered_spread_bps = (mid - offered) / mid * 10000
            mkt_label = "Market Bid Spread"
            off_label = "Your Offered Bid Spread"
            
        # Style based on action
        color_offered = "#008000" if action == "Buy" else "#cc0000"
        
        display(HTML(f"""
        <div style="background-color:#f9f9f9; padding: 15px; border-radius: 5px; border: 1px solid #ddd;">
            <h3 style="margin-top: 0;">Spread Dashboard</h3>
            <ul style="font-size:1.1em; line-height: 1.6;">
                <li><b>Mid Price:</b> <span style="color:#005580;">{mid:.4f}</span></li>
                <li><b>{mkt_label} (from Mid):</b> {mkt_spread_bps:.2f} bps</li>
                <li><b>{off_label} (from Mid):</b> <span style="color:{color_offered}; font-weight:bold;">{offered_spread_bps:.2f} bps</span></li>
            </ul>
        </div>
        """))

ask_val.observe(update_dashboard, names="value")
bid_val.observe(update_dashboard, names="value")
offered_val.observe(update_dashboard, names="value")
buy_sell_toggle.observe(update_dashboard, names="value")

ui = widgets.VBox([
    widgets.HBox([buy_sell_toggle, precise_mode_toggle], layout=widgets.Layout(margin="0 0 15px 0")),
    widgets.HTML("<h4>Current Market</h4>"),
    ask_ui,
    bid_ui,
    widgets.HTML("<h4>Your Offer</h4>"),
    offered_ui,
])

display(ui, out)
update_dashboard()


Output()